# train.ipynb — 최종 학습 (2차 평가 제출용)

**목적**: `05_tuning`에서 확정한 최종 설정으로 **train 전체 데이터(2022~2024)**를 재학습해 제출용 모델 파일을 만든다. 2차 평가 요건(`CLAUDE.md` 1절)에 따라 학습 코드(`train.ipynb`)와 추론 코드(`inference.ipynb`)를 분리한다 — 이 노트북은 오직 학습·모델 저장만 하고, 예측·제출 파일 생성은 `inference.ipynb`에서 한다.

**확정된 최종 설정** (`05_tuning.ipynb` / `reports/05_tuning.md` 근거):
- 모델 구조: 통합모델(3개 그룹을 `group_id` 범주형 피처로 구분해 하나의 모델로 학습, 타깃은 이용률=발전량/설비용량) + CatBoost
- 피처: `train_features_v2.parquet`의 50개 피처(`04_model_selection` 피처 선택 결과)
- 하이퍼파라미터: CatBoost 기본값(`learning_rate=0.05`, 나머지 기본) — Optuna 30 trial 튜닝 결과는 개선폭이 seed 표준편차보다 작아 채택하지 않음(05_tuning 8절)
- 손실 함수: 분위수 회귀(`Quantile:alpha=0.70`) — 채점 대상이 실측값 기준(`actual >= capacity*0.10`)이라는 `src/metric.py` 구조를 이용해 예측을 위쪽으로 겨냥(05_tuning 9절, 3-fold CV에서 +0.0305 개선, seed 표준편차의 47.6배로 안정적 확인)
- 표본 가중치: `actual`(kWh) — FICR이 실측값 가중합이라는 산식 구조를 학습에 반영
- 앙상블: seed 평균(`ensemble-final` 스킬 3절 — "seed 평균은 비용 대비 거의 항상 이득") — seed 3개(42/7/2024, 05_tuning에서 안정성 확인에 쓴 것과 동일)로 각각 전체 데이터를 재학습해 추론 시 평균

**입력**: `data/processed/train_features_v2.parquet`
**출력**: `models/catboost_seed{42,7,2024}.cbm`(각 seed 모델), `models/model_config.json`(재현에 필요한 모든 설정)

**early stopping 문제와 해결**: 전체 데이터로 재학습하면 검증셋이 없어 early stopping을 쓸 수 없다. `ensemble-final` 스킬 4절 표준 관행대로, 05_tuning과 같은 3-fold CV 구조에서 이 최종 설정으로 다시 학습해 각 fold가 early stopping으로 멈춘 반복 횟수(iteration)를 확인하고, 그 평균의 1.05배를 전체 데이터 재학습의 고정 `iterations`로 사용한다(검증셋이 없어졌으니 살짝 여유를 더 준다는 뜻).

## 0. 셋업

패키지를 불러오고 환경을 확인한다. `inference.ipynb`와 완전히 분리된 노트북이므로 필요한 import를 전부 이 노트북 안에서 다시 한다(`ensemble-final` 스킬 5절 — 두 노트북은 각각 단독으로 처음부터 끝까지 실행 가능해야 함).

In [1]:
import sys
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import catboost
from catboost import CatBoostRegressor

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import TARGET_COLS, CAPACITY_KWH

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

print("python:", sys.version)
print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("pandas:", pd.__version__, "| numpy:", np.__version__, "| catboost:", catboost.__version__)
print("OS:", platform.platform())

python: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
python executable: d:\공모전\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: d:\공모전\wind_forecast
pandas: 3.0.3 | numpy: 2.5.1 | catboost: 1.2.10
OS: Windows-11-10.0.26200-SP0


## 1. 데이터 로드 (train 전체)

`04_model_selection`/`05_tuning`에서 쓴 v2 피처셋(50개)을 그대로 불러온다. train/test가 동일 로직으로 만들어졌는지는 `01_preprocessing`~`03_features`에서 이미 확인했으므로 여기서는 shape과 기간만 재확인한다.

In [2]:
train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")

DROP_COLS = {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}
FEATURE_COLS = [c for c in train_features.columns if c not in DROP_COLS]

print("train_features:", train_features.shape)
print("피처 개수:", len(FEATURE_COLS))
print("기간:", train_features["forecast_kst_dtm"].min(), "~", train_features["forecast_kst_dtm"].max())

train_features: (26304, 54)
피처 개수: 50
기간: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


## 2. 최종 설정 확정

`05_tuning.ipynb`에서 확정한 값을 그대로 하드코딩한다(재현성 — 다른 곳에서 값을 가져오지 않고 이 노트북만 봐도 전체 설정을 알 수 있게).

In [3]:
FINAL_HYPERPARAMS = {"learning_rate": 0.05}  # CatBoost 기본값. 05_tuning 8절: Optuna 튜닝 효과는 seed 표준편차보다 작아 채택하지 않음
QUANTILE_ALPHA = 0.70           # 05_tuning 9절: 3-fold CV +0.0305 개선(seed 표준편차의 47.6배로 안정적). peak(0.71)보다 안정적인 0.70 채택
USE_SAMPLE_WEIGHT = True        # 학습 표본 가중치 = actual(kWh) — FICR이 실측값 가중합이라는 산식 구조 반영
ENSEMBLE_SEEDS = [42, 7, 2024]  # 05_tuning에서 안정성 확인에 쓴 seed와 동일 — 재현성 유지
EARLY_STOP_ROUNDS = 100
ITERATIONS_CAP = 2000           # early stopping 탐색 시 상한(05_tuning과 동일)

GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]

print("FINAL_HYPERPARAMS:", FINAL_HYPERPARAMS)
print("QUANTILE_ALPHA:", QUANTILE_ALPHA, "| USE_SAMPLE_WEIGHT:", USE_SAMPLE_WEIGHT)
print("ENSEMBLE_SEEDS:", ENSEMBLE_SEEDS)

FINAL_HYPERPARAMS: {'learning_rate': 0.05}
QUANTILE_ALPHA: 0.7 | USE_SAMPLE_WEIGHT: True
ENSEMBLE_SEEDS: [42, 7, 2024]


## 3. 최종 `iterations` 결정 — 3-fold CV로 early stopping 반복 횟수 확인

`ensemble-final` 스킬 4절: 전체 데이터로 재학습하면 검증셋이 사라져 early stopping을 못 쓴다 → **CV에서 관측된 최적 iteration의 평균(×1.05)으로 `iterations`를 고정**하는 게 표준 관행. `05_tuning`과 동일한 확장 윈도우 3-fold 구조로 이 최종 설정(τ=0.70, actual 가중, `FINAL_HYPERPARAMS`)을 다시 학습해서, 각 fold가 early stopping으로 멈춘 시점(`model.best_iteration_`)을 확인한다.

In [4]:
FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00"},
]


def make_fold_train(fold):
    dtm = train_features["forecast_kst_dtm"]
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])
    return train_features[(dtm >= train_start) & (dtm <= train_end)].reset_index(drop=True)


def to_long(df):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        sub["actual_kwh"] = sub[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + FEATURE_COLS])
    return pd.concat(frames, ignore_index=True)


features = FEATURE_COLS + ["group_id"]
best_iterations = []

for fold in FOLDS:
    fold_train = make_fold_train(fold)
    long_df = to_long(fold_train)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * 0.15)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    model = CatBoostRegressor(
        loss_function=f"Quantile:alpha={QUANTILE_ALPHA}",
        iterations=ITERATIONS_CAP,
        random_seed=42,
        verbose=False,
        **FINAL_HYPERPARAMS,
    )
    fit_kwargs = {"sample_weight": fit_rows["actual_kwh"].to_numpy()} if USE_SAMPLE_WEIGHT else {}
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], early_stopping_rounds=EARLY_STOP_ROUNDS, verbose=False,
        **fit_kwargs,
    )
    best_iterations.append(model.best_iteration_)
    print(f"{fold['name']}: best_iteration_={model.best_iteration_}")

mean_iter = float(np.mean(best_iterations))
FINAL_ITERATIONS = int(np.ceil(mean_iter * 1.05))
print(f"\n평균 best_iteration_: {mean_iter:.1f} -> 1.05배 적용 후 FINAL_ITERATIONS = {FINAL_ITERATIONS}")

fold1: best_iteration_=356
fold2: best_iteration_=518
fold3: best_iteration_=1170

평균 best_iteration_: 681.3 -> 1.05배 적용 후 FINAL_ITERATIONS = 716


## 4. 전체 데이터 재학습 (seed 앙상블)

`ensemble-final` 스킬 4절대로 **train 전체(2022~2024)**를 그대로 학습에 쓴다(더 이상 fold로 나누지 않음 — 검증이 아니라 최종 제출용 모델이므로). `ENSEMBLE_SEEDS` 각각으로 독립 학습해 `models/`에 저장한다. `iterations`는 3절에서 정한 `FINAL_ITERATIONS`로 고정하고 early stopping은 쓰지 않는다(검증셋이 없으므로).

In [5]:
full_long = to_long(train_features)
full_long["group_id"] = pd.Categorical(full_long["group_id"], categories=GROUP_ID_CATEGORIES)

fit_kwargs_full = {"sample_weight": full_long["actual_kwh"].to_numpy()} if USE_SAMPLE_WEIGHT else {}

trained_models = {}
for seed in ENSEMBLE_SEEDS:
    model = CatBoostRegressor(
        loss_function=f"Quantile:alpha={QUANTILE_ALPHA}",
        iterations=FINAL_ITERATIONS,
        random_seed=seed,
        verbose=False,
        **FINAL_HYPERPARAMS,
    )
    model.fit(
        full_long[features], full_long["utilization"],
        cat_features=["group_id"], verbose=False,
        **fit_kwargs_full,
    )
    model_path = MODELS_DIR / f"catboost_seed{seed}.cbm"
    model.save_model(str(model_path))
    trained_models[seed] = model
    print(f"seed={seed} 학습 완료, 저장: {model_path}")

seed=42 학습 완료, 저장: d:\공모전\wind_forecast\models\catboost_seed42.cbm
seed=7 학습 완료, 저장: d:\공모전\wind_forecast\models\catboost_seed7.cbm
seed=2024 학습 완료, 저장: d:\공모전\wind_forecast\models\catboost_seed2024.cbm


## 5. 재현용 설정 저장

`inference.ipynb`가 이 노트북과 독립적으로 실행되면서도 같은 설정(피처 목록, 시드, 손실 함수, 그룹 매핑 등)을 그대로 쓸 수 있도록 `models/model_config.json`에 저장한다.

In [6]:
model_config = {
    "feature_cols": FEATURE_COLS,
    "quantile_alpha": QUANTILE_ALPHA,
    "use_sample_weight": USE_SAMPLE_WEIGHT,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "final_iterations": FINAL_ITERATIONS,
    "hyperparams": FINAL_HYPERPARAMS,
    "group_id_map": GROUP_ID_MAP,
    "capacity_kwh": CAPACITY_KWH,
    "catboost_version": catboost.__version__,
}

config_path = MODELS_DIR / "model_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(model_config, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {config_path}")
model_config

저장 완료: d:\공모전\wind_forecast\models\model_config.json


{'feature_cols': ['gfs_ws10m',
  'gfs_ws80m',
  'gfs_ws100m',
  'gfs_wd100m_sin',
  'gfs_wd100m_cos',
  'gfs_ws100m_sq',
  'gfs_ws100m_cube',
  'gfs_ws850hpa',
  'kpx_group_1_ldaps_ws10m',
  'kpx_group_1_ldaps_ws10m_sq',
  'kpx_group_1_ldaps_ws10m_cube',
  'kpx_group_1_ldaps_wd10m_sin',
  'kpx_group_1_ldaps_wd10m_cos',
  'kpx_group_2_ldaps_ws10m',
  'kpx_group_2_ldaps_ws10m_sq',
  'kpx_group_2_ldaps_ws10m_cube',
  'kpx_group_2_ldaps_wd10m_sin',
  'kpx_group_2_ldaps_wd10m_cos',
  'kpx_group_3_ldaps_ws10m',
  'kpx_group_3_ldaps_ws10m_sq',
  'kpx_group_3_ldaps_ws10m_cube',
  'kpx_group_3_ldaps_wd10m_sin',
  'kpx_group_3_ldaps_wd10m_cos',
  'gfs_rho',
  'gfs_ws100m_rho_corrected',
  'kpx_group_1_ldaps_rho',
  'kpx_group_1_ldaps_ws10m_rho_corrected',
  'kpx_group_2_ldaps_rho',
  'kpx_group_2_ldaps_ws10m_rho_corrected',
  'kpx_group_3_ldaps_rho',
  'kpx_group_3_ldaps_ws10m_rho_corrected',
  'gfs_temp_diff_2m_850hpa',
  'kpx_group_1_ldaps_blh',
  'kpx_group_1_ldaps_gust_range_50m',
  'kpx_gro

## 6. Sanity check — 재학습 모델의 예측이 그럴듯한가

재학습 모델은 검증셋이 없어 Score를 낼 수 없다(`ensemble-final` 스킬 4절). 대신 **train 데이터 자체에 대한 예측(in-sample)이 실제 분포와 형태가 비슷한지**로 명백한 실수(피처 순서 오류, 그룹 매핑 오류 등)만 걸러낸다. 이건 성능 검증이 아니라 "완전히 잘못되지는 않았는지" 확인하는 최소한의 안전장치라는 점에 유의.

In [7]:
sanity_pred = {}
for g in TARGET_COLS:
    g_rows = train_features.copy()
    g_rows["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(g_rows), categories=GROUP_ID_CATEGORIES)
    seed_preds = [
        np.clip(trained_models[seed].predict(g_rows[features]) * CAPACITY_KWH[g], 0, CAPACITY_KWH[g])
        for seed in ENSEMBLE_SEEDS
    ]
    sanity_pred[g] = np.mean(seed_preds, axis=0)

sanity_df = train_features[["forecast_kst_dtm"]].copy()
for g in TARGET_COLS:
    sanity_df[f"{g}_actual"] = train_features[g]
    sanity_df[f"{g}_pred"] = sanity_pred[g]

sanity_df["month"] = sanity_df["forecast_kst_dtm"].dt.month
monthly_cols = [f"{g}_actual" for g in TARGET_COLS] + [f"{g}_pred" for g in TARGET_COLS]
monthly_compare = sanity_df.groupby("month")[monthly_cols].mean()
print("월별 평균 발전량(kWh) — 실제 vs 예측(in-sample, seed 평균):")
print(monthly_compare)

for g in TARGET_COLS:
    at_zero = (sanity_pred[g] <= 1e-6).mean()
    at_capacity = (sanity_pred[g] >= CAPACITY_KWH[g] - 1e-6).mean()
    print(f"{g}: 0kWh 근접 비율={at_zero:.1%}, 설비용량 근접 비율={at_capacity:.1%}")

월별 평균 발전량(kWh) — 실제 vs 예측(in-sample, seed 평균):
       kpx_group_1_actual  kpx_group_2_actual  kpx_group_3_actual  \
month                                                               
1             9355.272067         9685.125389         8498.592399   
2             6844.852532         6980.224835         4592.887846   
3             7098.556801         7681.636097         6601.090220   
4             6908.929998         7510.088488         5483.716352   
5             7052.539037         7259.455513         5449.645955   
6             4931.340902         5314.715026         3593.964154   
7             5637.695787         6207.910215         7521.339821   
8             3975.495720         4757.938298         2489.548800   
9             3184.124461         3612.100022         2046.757793   
10            4830.970539         5040.188527         3219.811920   
11            8142.289175         8170.425380         7603.051088   
12           11339.549944        12492.960864         94

**해석 — 예측이 실제보다 높은 건 버그가 아니라 의도한 설계다**: 위 표를 보면 모든 월에서 `_pred`가 `_actual`보다 15~25%가량 높게 나온다. 언뜻 "모델이 발전량을 과대예측하는 문제"처럼 보일 수 있지만, **이는 05_tuning 9절에서 확정한 분위수 회귀(τ=0.70)가 의도한 그대로의 결과**다 — τ=0.70은 "중앙값(50%)"이 아니라 "이 값보다 클 확률이 30%인 지점(70분위수)"을 예측하도록 학습시킨 것이므로, 평균/중앙값보다 체계적으로 높게 나오는 것이 정상이다. 모든 그룹·모든 월에서 비슷한 비율로 높게 나온다는 점(특정 월만 튀지 않음)이 "설계대로 작동한다"는 근거다 — 만약 버그였다면 특정 그룹·특정 월에서만 이상하게 튀거나, 계절 패턴 자체가 실제와 반대로 나오는 등 불규칙한 모습을 보였을 것이다.

**클리핑 비율도 정상 범위**: 0kWh 근접 비율이 0.2~1.5%, 설비용량 근접 비율이 0%로 극단값에 몰린 예측이 거의 없다 — 명백한 오류(예: 피처 순서가 뒤바뀌어 예측이 이상하게 포화되는 등)의 흔적은 보이지 않는다.

**fold별 `best_iteration_` 참고**: fold1=356, fold2=518, fold3=1170으로 fold3가 유독 크게 나왔다(학습 데이터가 가장 많은 fold라 더 많은 트리가 필요했던 것으로 추정). 평균(681.3)의 1.05배인 716을 최종 `iterations`로 썼는데, fold간 편차가 3배 넘게 크다는 점은 참고해두자 — 만약 나중에 재학습 결과가 이상하면 이 편차가 원인일 수 있다는 뜻으로, `FINAL_ITERATIONS`를 좀 더 크게(예: fold3 값에 가깝게) 잡는 대안도 검토 가능하다.

## 요약

- `05_tuning`에서 확정한 설정(통합모델 CatBoost + 기본 하이퍼파라미터 + 분위수 회귀 τ=0.70 + actual 가중)으로 **train 전체(2022~2024)**를 재학습했다
- early stopping이 불가능해진 문제는 3-fold CV의 `best_iteration_` 평균×1.05로 `iterations`를 고정하는 표준 관행으로 해결했다(`ensemble-final` 스킬 4절)
- seed 3개(42/7/2024) 앙상블로 학습해 `models/catboost_seed{seed}.cbm`에 저장했다 — 추론 시 이 3개의 예측을 평균한다(`ensemble-final` 스킬 3절 "seed 평균은 비용 대비 거의 항상 이득")
- 재현에 필요한 모든 설정을 `models/model_config.json`에 저장해 `inference.ipynb`가 독립적으로 실행 가능하게 했다
- Sanity check로 월별 평균 발전량과 클리핑 비율을 확인했다(성능 검증 아님 — 명백한 오류만 거르는 용도)

**다음**: `inference.ipynb`에서 이 모델들과 `test_features_v2.parquet`으로 예측 → 후처리 → `validate_submission()` → `submissions/`에 제출 파일 저장